In [3]:
from os.path import join
import pandas as pd
import pandas as pd
import numpy as np
import platform

if platform.node() == 'progress':
    PROJECT_ROOT = '/cim/zahrat/workshop/dent'
else:
    PROJECT_ROOT = '/usr/local/data/zahrat/workshop/dent'


In [4]:
df = pd.read_csv(join(PROJECT_ROOT, 'data/chexpert/finetune.csv'))
df.head()

df = df[(df['Frontal/Lateral'] == 'Frontal')]
df['Pleural Effusion'] = np.where(df['No Finding'] == 1, 0, df['Pleural Effusion'])
df.fillna(0, inplace=True)

df = df.dropna(subset=['Support Devices', 'Pleural Effusion'])

df00 = df[(df['Pleural Effusion'] == 0) & (df['Support Devices'] == 0)].sample(frac=1)
df01 = df[(df['Support Devices'] == 0) & (df['Pleural Effusion'] == 1)].sample(frac=1)
df10 = df[(df['Support Devices'] == 1) & (df['Pleural Effusion'] == 0)].sample(frac=1)
df11 = df[(df['Support Devices'] == 1) & (df['Pleural Effusion'] == 1)].sample(frac=1)

for df in [df00, df01, df10, df11]:
    print(len(df))


51783
26643
52234
49917


In [5]:
nsample_per_class_test = 1000
test_df = pd.concat([df00[:nsample_per_class_test], df01[:nsample_per_class_test], df10[:nsample_per_class_test], df11[:nsample_per_class_test]])

nsample_majority = min(len(df00), len(df11)) - nsample_per_class_test
nsample_minority = nsample_majority // 10

train_val_ratio = 0.8

nsample_majority_train = int(nsample_majority * train_val_ratio)
nsample_minority_train = int(nsample_minority * train_val_ratio)

nsample_majority_val = nsample_majority - nsample_majority_train
nsample_minority_val = nsample_minority - nsample_minority_train

df00_train = df00[nsample_per_class_test:nsample_per_class_test + nsample_majority_train]
df01_train = df01[nsample_per_class_test:nsample_per_class_test + nsample_minority_train]
df10_train = df10[nsample_per_class_test:nsample_per_class_test + nsample_minority_train]
df11_train = df11[nsample_per_class_test:nsample_per_class_test + nsample_majority_train]

df00_val = df00[nsample_per_class_test + nsample_majority_train:nsample_per_class_test + nsample_majority_train + nsample_majority_val]
df01_val = df01[nsample_per_class_test + nsample_minority_train:nsample_per_class_test + nsample_minority_train + nsample_minority_val]
df10_val = df10[nsample_per_class_test + nsample_minority_train:nsample_per_class_test + nsample_minority_train + nsample_minority_val]
df11_val = df11[nsample_per_class_test + nsample_majority_train:nsample_per_class_test + nsample_majority_train + nsample_majority_val]

train_df = pd.concat([df00_train, df01_train, df10_train, df11_train]).sample(frac=1)
val_df = pd.concat([df00_val, df01_val, df10_val, df11_val]).sample(frac=1)

for df in [train_df, val_df, test_df]:
    print(len(df))
    
train_df.to_csv(join(PROJECT_ROOT, 'data/chexpert/imbalanced_pe_sd/chexpert_train_imbalanced.csv'), index=False)
val_df.to_csv(join(PROJECT_ROOT, 'data/chexpert/imbalanced_pe_sd/chexpert_val_imbalanced.csv'), index=False)
test_df.to_csv(join(PROJECT_ROOT, 'data/chexpert/imbalanced_pe_sd/chexpert_test_balanced.csv'), index=False)


86090
21526
4000


In [20]:
test_df[(test_df['Support Devices'] == 0) & (test_df['Pleural Effusion'] == 0)]

,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
106461,CheXpert-v1.0_512x512/train/patient32247/study...,Male,74,Frontal,AP,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0
29344,CheXpert-v1.0_512x512/train/patient09081/study...,Male,55,Frontal,AP,0.0,0.0,0.0,1.0,0.0,0.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0
45247,CheXpert-v1.0_512x512/train/patient13847/study...,Male,53,Frontal,PA,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14728,CheXpert-v1.0_512x512/train/patient04652/study...,Female,78,Frontal,AP,0.0,0.0,0.0,1.0,1.0,0.0,-1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
31686,CheXpert-v1.0_512x512/train/patient09802/study...,Female,56,Frontal,AP,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157080,CheXpert-v1.0_512x512/train/patient44824/study...,Male,88,Frontal,AP,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0
38291,CheXpert-v1.0_512x512/train/patient11710/study...,Male,68,Frontal,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
99861,CheXpert-v1.0_512x512/train/patient30208/study...,Female,82,Frontal,AP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
182758,CheXpert-v1.0_512x512/train/patient58015/study...,Male,40,Frontal,AP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
test_df_imbalanced = pd.concat([df00[:1000], df01[:250], df10[:250], df11[:1000]])
test_df_imbalanced.to_csv(join(PROJECT_ROOT, 'data/chexpert/imbalanced_pe_sd/chexpert_test_imbalanced.csv'), index=False)